# 09 — Database Extraction (SQLite Dev)

**C1 Source Type:** `Base de données`

---

## Objective

Demonstrate data extraction from a **relational database** — the fifth C1 source type.
We use **SQLite** (the dev/CI dialect, per ADR-0004) with the same schema as the production
Neon PostgreSQL database defined in `docs/architecture/data-design.md`.

### What this notebook does

1. **Create** the Sicurre SQLite dev database with the canonical schema
2. **Load** French phishing data from upstream notebooks:
   - `adapted_fr_phishing.csv` (notebook 10 — EN→FR cultural adaptation)
   - `synthetic_fr_emails.csv` (notebook 11 — synthetic generation)
3. **Seed** users + model versions with `Faker(fr_FR)` for realistic metadata
4. **Extract** data using SQL queries (aggregations, JOINs, filtering)
5. **Export** query results to CSV

### Data Pipeline

```
combined_final_clean.csv (113K EN)     Faker(fr_FR) templates
            │                                   │
     ┌──────┴──────┐                    ┌───────┴───────┐
     │ Notebook 10 │                    │  Notebook 11  │
     │  EN→FR      │                    │  Synthetic FR │
     └──────┬──────┘                    └───────┬───────┘
            │                                   │
   adapted_fr_phishing.csv           synthetic_fr_emails.csv
            │                                   │
            └───────────────┬───────────────────┘
                            │
                   ┌────────┴────────┐
                   │  Notebook 09    │
                   │  SQLite DB      │
                   │  SQL Queries    │
                   └────────┬────────┘
                            │
                  data/raw/db/sicurre_dev.db
```

### Why SQLite?

| Criterion | Detail |
|-----------|--------|
| ADR-0004 | SQLite for dev/CI, Neon PostgreSQL for prod |
| Dialect abstraction | SQLAlchemy ORM — same models, different backend |
| No credentials needed | File-based, no connection string |
| C1 requirement | Satisfies *"base de données"* extraction type |

### Output

- `data/raw/db/sicurre_dev.db` — SQLite database file
- `data/raw/db/db_threat_export_<N>_<date>.csv` — Extracted data

In [ ]:
# ── Imports & Constants ──────────────────────────────────────────────
from __future__ import annotations

import json
import random
import re
import uuid
from datetime import datetime, timezone, timedelta
from pathlib import Path

import pandas as pd
from faker import Faker
from sqlalchemy import (
    create_engine,
    Column,
    String,
    Float,
    DateTime,
    Text,
    Integer,
    UniqueConstraint,
    ForeignKey,
    text as sa_text,
)
from sqlalchemy.orm import DeclarativeBase, sessionmaker

# ── Configuration ────────────────────────────────────────────────────
DB_DIR: Path = Path("data/raw/db")
DB_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH: Path = DB_DIR / "sicurre_dev.db"
DB_URL: str = f"sqlite:///{DB_PATH}"

OUTPUT_DIR: Path = DB_DIR

# Upstream CSVs from notebooks 10 & 11
ADAPTED_CSV: Path = DB_DIR / "adapted_fr_phishing.csv"
SYNTHETIC_CSV: Path = DB_DIR / "synthetic_fr_emails.csv"

SEED: int = 42
random.seed(SEED)
fake = Faker("fr_FR")
Faker.seed(SEED)

print(f"Database     : {DB_PATH.resolve()}")
print(f"SQLAlchemy   : {DB_URL}")
print(f"Output dir   : {OUTPUT_DIR.resolve()}")
print(f"Adapted CSV  : {ADAPTED_CSV} (exists: {ADAPTED_CSV.exists()})")
print(f"Synthetic CSV: {SYNTHETIC_CSV} (exists: {SYNTHETIC_CSV.exists()})")

Database     : /Users/michaeladebayo/Documents/Simplon/brief_projects/sicurre/data/raw/db/sicurre_dev.db
SQLAlchemy   : sqlite:///data/raw/db/sicurre_dev.db
Output dir   : /Users/michaeladebayo/Documents/Simplon/brief_projects/sicurre/data/raw/db
Adapted CSV  : data/raw/db/adapted_fr_phishing.csv (exists: True)
Synthetic CSV: data/raw/db/synthetic_fr_emails.csv (exists: True)


## 1. Define Schema (SQLAlchemy ORM)

These models mirror the canonical schema from `docs/architecture/data-design.md`.
The same classes work against both SQLite (dev) and PostgreSQL (prod) via
SQLAlchemy dialect abstraction.

In [2]:
# ── ORM Models ────────────────────────────────────────────────────────

class Base(DeclarativeBase):
    pass


class User(Base):
    __tablename__ = "users"
    
    id = Column(String, primary_key=True, default=lambda: str(uuid.uuid4()))
    email = Column(String, nullable=False, unique=True)
    display_name = Column(String)
    plan = Column(String, nullable=False, default="free")
    created_at = Column(DateTime, default=lambda: datetime.now(timezone.utc))
    updated_at = Column(DateTime)


class ThreatLog(Base):
    __tablename__ = "threat_log"
    __table_args__ = (
        UniqueConstraint("user_id", "message_id", name="uq_user_message"),
    )
    
    id = Column(String, primary_key=True, default=lambda: str(uuid.uuid4()))
    user_id = Column(String, ForeignKey("users.id"), nullable=False)
    message_id = Column(String, nullable=False)
    subject = Column(Text)  # Email subject line
    body_preview = Column(Text)  # Anonymized preview (PII redacted)
    received_at = Column(DateTime)
    verdict = Column(String, nullable=False)  # phishing | legitimate
    confidence = Column(Float)
    signals = Column(Text)  # JSON string
    archetype = Column(String)  # French phishing archetype
    source_dataset = Column(String)  # adapted_en_fr | synthetic_fr
    model_version = Column(String, nullable=False)
    action_taken = Column(String)  # trashed | none | restored
    action_at = Column(DateTime)
    created_at = Column(DateTime, default=lambda: datetime.now(timezone.utc))


class Feedback(Base):
    __tablename__ = "feedback"
    __table_args__ = (
        UniqueConstraint("threat_log_id", "user_id", name="uq_feedback_threat_user"),
    )
    
    id = Column(String, primary_key=True, default=lambda: str(uuid.uuid4()))
    threat_log_id = Column(String, ForeignKey("threat_log.id"), nullable=False)
    user_id = Column(String, ForeignKey("users.id"), nullable=False)
    feedback_label = Column(String, nullable=False)
    comment = Column(Text)
    created_at = Column(DateTime, default=lambda: datetime.now(timezone.utc))


class ModelVersion(Base):
    __tablename__ = "model_versions"
    
    id = Column(String, primary_key=True, default=lambda: str(uuid.uuid4()))
    version_tag = Column(String, nullable=False, unique=True)
    artifact_uri = Column(String)
    f1_score = Column(Float)
    precision_score = Column(Float)
    recall_score = Column(Float)
    eval_samples = Column(Integer)
    promoted_at = Column(DateTime)
    created_at = Column(DateTime, default=lambda: datetime.now(timezone.utc))


print("ORM models defined: User, ThreatLog, Feedback, ModelVersion")
print(f"Tables: {[cls.__tablename__ for cls in [User, ThreatLog, Feedback, ModelVersion]]}")

ORM models defined: User, ThreatLog, Feedback, ModelVersion
Tables: ['users', 'threat_log', 'feedback', 'model_versions']


In [3]:
# ── Create Database & Tables ──────────────────────────────────────────

# Remove old DB if it exists (clean re-seed)
if DB_PATH.exists():
    DB_PATH.unlink()
    print(f"Removed old database: {DB_PATH}")

engine = create_engine(DB_URL, echo=False)
Base.metadata.create_all(engine)

SessionLocal = sessionmaker(bind=engine)

# Verify tables exist
with engine.connect() as conn:
    tables = conn.execute(sa_text("SELECT name FROM sqlite_master WHERE type='table'")).fetchall()
    print(f"Tables created: {[t[0] for t in tables]}")
    
print(f"Database file: {DB_PATH} ({DB_PATH.stat().st_size / 1024:.1f} KB)")

Tables created: ['users', 'model_versions', 'threat_log', 'feedback']
Database file: data/raw/db/sicurre_dev.db (52.0 KB)


## 2. Load Upstream Data

Load the French phishing data produced by notebooks 10 and 11.
These CSVs contain culturally-adapted and synthetic French emails.

In [4]:
# ── Load adapted + synthetic CSVs ─────────────────────────────────────

dfs: list[pd.DataFrame] = []

if ADAPTED_CSV.exists():
    df_adapted: pd.DataFrame = pd.read_csv(ADAPTED_CSV)
    dfs.append(df_adapted)
    print(f"Adapted CSV  : {len(df_adapted):,} rows")
else:
    print(f"⚠ Adapted CSV not found: {ADAPTED_CSV}")
    print("  → Run notebook 10 first!")

if SYNTHETIC_CSV.exists():
    df_synthetic: pd.DataFrame = pd.read_csv(SYNTHETIC_CSV)
    dfs.append(df_synthetic)
    print(f"Synthetic CSV: {len(df_synthetic):,} rows")
else:
    print(f"⚠ Synthetic CSV not found: {SYNTHETIC_CSV}")
    print("  → Run notebook 11 first!")

if not dfs:
    raise FileNotFoundError(
        "No upstream CSVs found. Run notebooks 10 and 11 first."
    )

df_emails: pd.DataFrame = pd.concat(dfs, ignore_index=True)
print(f"\nCombined     : {len(df_emails):,} emails")
print(f"Labels       : {dict(df_emails['label'].value_counts())}")
print(f"Sources      : {dict(df_emails['source'].value_counts())}")

Adapted CSV  : 2,400 rows
Synthetic CSV: 2,863 rows

Combined     : 5,263 emails
Labels       : {1: np.int64(4400), 0: np.int64(863)}
Sources      : {'synthetic_fr': np.int64(2863), 'adapted_en_fr': np.int64(2400)}


## 3. Seed Database with Faker Users + Upstream Data

We create **10 realistic French auto-entrepreneur users** with `Faker(fr_FR)`
and distribute the upstream emails across them as threat log entries.

In [5]:
# ── Seed users (Faker fr_FR) ──────────────────────────────────────────
now = datetime.now(timezone.utc)

N_USERS: int = 10
plans: list[str] = ["free"] * 4 + ["pro"] * 4 + ["business"] * 2

users_data: list[dict] = []
for i in range(N_USERS):
    first: str = fake.first_name()
    last: str = fake.last_name()
    domain: str = random.choice(["gmail.com", "outlook.fr", "yahoo.fr", "orange.fr", "free.fr"])
    users_data.append({
        "email": f"{first.lower()}.{last.lower()}@{domain}",
        "display_name": f"{first} {last}",
        "plan": plans[i],
    })

# ── Seed model versions ───────────────────────────────────────────────
models_data: list[dict] = [
    {"version_tag": "v0.1.0", "f1_score": 0.89, "precision_score": 0.91, "recall_score": 0.87, "eval_samples": 500},
    {"version_tag": "v0.2.0", "f1_score": 0.93, "precision_score": 0.94, "recall_score": 0.92, "eval_samples": 1200},
    {"version_tag": "v0.3.0", "f1_score": 0.96, "precision_score": 0.97, "recall_score": 0.95, "eval_samples": 2000},
]

# ── Phishing signal patterns ──────────────────────────────────────────
phishing_signals: list[list[str]] = [
    ["DMARC fail", "Suspicious URL"],
    ["SPF fail", "Urgency language"],
    ["Homograph domain", "Credential request"],
    ["URL mismatch", "DKIM fail"],
    ["New sender", "Attachment suspicious"],
    ["Reply-to mismatch", "French impersonation"],
    ["Lookalike domain", "PII request"],
]

print(f"Users to seed   : {len(users_data)} ({dict(pd.Series(plans).value_counts())})")
for u in users_data:
    print(f"  {u['display_name']:25s} ({u['plan']}) — {u['email']}")
print(f"\nModel versions  : {len(models_data)}")
print(f"Signal patterns : {len(phishing_signals)}")

Users to seed   : 10 ({'free': np.int64(4), 'pro': np.int64(4), 'business': np.int64(2)})
  Lucie Marie               (free) — lucie.marie@gmail.com
  André Traore              (free) — andré.traore@gmail.com
  Robert Moulin             (free) — robert.moulin@yahoo.fr
  Maurice Joly              (free) — maurice.joly@outlook.fr
  Nathalie Henry            (pro) — nathalie.henry@outlook.fr
  Margaret Traore           (pro) — margaret.traore@outlook.fr
  Émilie Fernandez          (pro) — émilie.fernandez@gmail.com
  Jeanne Briand             (pro) — jeanne.briand@free.fr
  Arthur Martinez           (business) — arthur.martinez@gmail.com
  Éric Carpentier           (business) — éric.carpentier@free.fr

Model versions  : 3
Signal patterns : 7


In [6]:
# ── Insert seed data ──────────────────────────────────────────────────

with SessionLocal() as session:
    # ── Users ─────────────────────────────────────────────────────────
    user_ids: list[str] = []
    for ud in users_data:
        user = User(**ud)
        session.add(user)
        session.flush()
        user_ids.append(user.id)
    
    # ── Model versions ────────────────────────────────────────────────
    for md in models_data:
        mv = ModelVersion(
            **md,
            artifact_uri=f"hf://sicurre/camembertv2-phishing-fr:{md['version_tag']}"
        )
        session.add(mv)
    
    # ── Threat log entries (from upstream email data) ──────────────────
    model_tags: list[str] = [m["version_tag"] for m in models_data]
    threat_ids: list[str] = []
    
    for idx, row in df_emails.iterrows():
        user_id: str = random.choice(user_ids)
        is_phishing: bool = int(row["label"]) == 1
        confidence: float = round(
            random.uniform(0.80, 0.99) if is_phishing else random.uniform(0.50, 0.80),
            3
        )
        
        # Extract subject from text ("Objet : ...\n\n...") if present
        text: str = str(row.get("text", ""))
        subject: str = ""
        body_preview: str = text[:200]
        if text.startswith("Objet : "):
            parts = text.split("\n\n", 1)
            subject = parts[0].replace("Objet : ", "", 1)
            body_preview = parts[1][:200] if len(parts) > 1 else text[:200]
        
        # Anonymize PII in preview (RGPD compliance)
        body_preview = re.sub(
            r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',
            '[EMAIL]', body_preview
        )
        body_preview = re.sub(
            r'\b0[1-9][ .-]?(?:\d{2}[ .-]?){4}\b',
            '[PHONE]', body_preview
        )
        
        threat = ThreatLog(
            user_id=user_id,
            message_id=f"msg_{uuid.uuid4().hex[:12]}",
            subject=subject,
            body_preview=body_preview,
            received_at=now - timedelta(
                days=random.randint(0, 90),
                hours=random.randint(0, 23),
                minutes=random.randint(0, 59),
            ),
            verdict="phishing" if is_phishing else "legitimate",
            confidence=confidence,
            signals=json.dumps(
                random.choice(phishing_signals) if is_phishing else []
            ),
            archetype=str(row.get("archetype", "")),
            source_dataset=str(row.get("source", "")),
            model_version=random.choice(model_tags),
            action_taken="trashed" if is_phishing and confidence > 0.85 else "none",
        )
        if threat.action_taken == "trashed":
            threat.action_at = threat.received_at + timedelta(
                seconds=random.uniform(0.5, 2.0)
            )
        
        session.add(threat)
        session.flush()
        threat_ids.append(threat.id)
    
    # ── Feedback (10% of threats) ─────────────────────────────────────
    feedback_count: int = max(30, len(threat_ids) // 10)
    feedback_threats: list[str] = random.sample(
        threat_ids, min(feedback_count, len(threat_ids))
    )
    for tid in feedback_threats:
        fb = Feedback(
            threat_log_id=tid,
            user_id=random.choice(user_ids),
            feedback_label=random.choice([
                "true_positive", "false_positive",
                "false_negative", "true_negative",
            ]),
            comment=random.choice([
                None,
                "Email légitime de ma banque",
                "C'était bien du phishing URSSAF",
                "Fausse alerte — newsletter Simplon",
                "Hameçonnage DGFiP confirmé",
                "Mon client m'a envoyé cette facture",
                "Phishing Ameli classique",
            ]),
        )
        session.add(fb)
    
    session.commit()

print(f"Seeded:")
print(f"  Users           : {len(user_ids)}")
print(f"  Model versions  : {len(models_data)}")
print(f"  Threat log      : {len(threat_ids):,}")
print(f"  Feedback        : {len(feedback_threats)}")
print(f"  Database size   : {DB_PATH.stat().st_size / 1024:.1f} KB")

Seeded:
  Users           : 10
  Model versions  : 3
  Threat log      : 5,263
  Feedback        : 526
  Database size   : 3788.0 KB


## 4. Extract Data via SQL Queries

Now we demonstrate **database extraction** — the actual C1 requirement.
We run analytical queries against the Sicurre database to extract insights
from the French phishing data.

In [7]:
# ── Query 1: Threat distribution by verdict ──────────────────────────
query_1: str = """
SELECT
    verdict,
    COUNT(*) AS count,
    ROUND(AVG(confidence), 3) AS avg_confidence,
    MIN(confidence) AS min_confidence,
    MAX(confidence) AS max_confidence
FROM threat_log
GROUP BY verdict
ORDER BY count DESC
"""

df_q1: pd.DataFrame = pd.read_sql(query_1, engine)
print("Query 1: Threat distribution by verdict")
display(df_q1)

Query 1: Threat distribution by verdict


,verdict,count,avg_confidence,min_confidence,max_confidence
0,phishing,4400,0.894,0.8,0.99
1,legitimate,863,0.653,0.5,0.80


In [8]:
# ── Query 2: Per-user threat summary with plan context ───────────────
query_2: str = """
SELECT
    u.display_name,
    u.plan,
    COUNT(t.id) AS total_threats,
    SUM(CASE WHEN t.verdict = 'phishing' THEN 1 ELSE 0 END) AS phishing_count,
    SUM(CASE WHEN t.verdict = 'legitimate' THEN 1 ELSE 0 END) AS legit_count,
    SUM(CASE WHEN t.action_taken = 'trashed' THEN 1 ELSE 0 END) AS auto_trashed,
    ROUND(AVG(t.confidence), 3) AS avg_confidence
FROM users u
LEFT JOIN threat_log t ON u.id = t.user_id
GROUP BY u.id
ORDER BY total_threats DESC
"""

df_q2: pd.DataFrame = pd.read_sql(query_2, engine)
print("Query 2: Per-user threat summary")
display(df_q2)

Query 2: Per-user threat summary


,display_name,plan,total_threats,phishing_count,legit_count,auto_trashed,avg_confidence
0,Robert Moulin,free,577,486,91,330,0.852
1,Émilie Fernandez,pro,562,476,86,342,0.856
2,Jeanne Briand,pro,539,457,82,334,0.857
3,Éric Carpentier,business,534,446,88,318,0.855
4,Lucie Marie,free,526,434,92,316,0.853
5,André Traore,free,522,438,84,322,0.854
6,Maurice Joly,free,513,429,84,314,0.853
7,Margaret Traore,pro,512,420,92,302,0.852
8,Nathalie Henry,pro,490,402,88,300,0.855
9,Arthur Martinez,business,488,412,76,312,0.856


In [9]:
# ── Query 3: French phishing archetype distribution ──────────────────
query_3: str = """
SELECT
    archetype,
    source_dataset,
    COUNT(*) AS count,
    ROUND(AVG(confidence), 3) AS avg_confidence,
    SUM(CASE WHEN action_taken = 'trashed' THEN 1 ELSE 0 END) AS auto_trashed
FROM threat_log
WHERE verdict = 'phishing' AND archetype != ''
GROUP BY archetype, source_dataset
ORDER BY count DESC
"""

df_q3: pd.DataFrame = pd.read_sql(query_3, engine)
print("Query 3: French phishing archetype distribution")
display(df_q3)

Query 3: French phishing archetype distribution


,archetype,source_dataset,count,avg_confidence,auto_trashed
0,ameli_sante,adapted_en_fr,300,0.896,229
1,banque_securite,adapted_en_fr,300,0.891,205
2,caf_allocation,adapted_en_fr,300,0.891,206
3,dgfip_tax,adapted_en_fr,300,0.890,214
4,facture_paiement,adapted_en_fr,300,0.892,215
5,franceconnect_id,adapted_en_fr,300,0.890,216
6,laposte_colis,adapted_en_fr,300,0.897,227
7,urssaf_cotisation,adapted_en_fr,300,0.899,224
8,dgfip_tax,synthetic_fr,273,0.891,192
9,banque_securite,synthetic_fr,268,0.896,196


In [10]:
# ── Query 4: Model version performance vs user feedback ──────────────
query_4: str = """
SELECT
    t.model_version,
    mv.f1_score,
    COUNT(t.id) AS classifications,
    ROUND(AVG(t.confidence), 3) AS avg_confidence,
    SUM(CASE WHEN f.feedback_label = 'false_positive' THEN 1 ELSE 0 END) AS false_positives,
    SUM(CASE WHEN f.feedback_label = 'false_negative' THEN 1 ELSE 0 END) AS false_negatives,
    SUM(CASE WHEN f.feedback_label = 'true_positive' THEN 1 ELSE 0 END) AS true_positives
FROM threat_log t
LEFT JOIN model_versions mv ON t.model_version = mv.version_tag
LEFT JOIN feedback f ON t.id = f.threat_log_id
GROUP BY t.model_version
ORDER BY mv.f1_score DESC
"""

df_q4: pd.DataFrame = pd.read_sql(query_4, engine)
print("Query 4: Model version performance vs user feedback")
display(df_q4)

Query 4: Model version performance vs user feedback


,model_version,f1_score,classifications,avg_confidence,false_positives,false_negatives,true_positives
0,v0.3.0,0.96,1775,0.852,43,39,45
1,v0.2.0,0.93,1709,0.856,44,37,40
2,v0.1.0,0.89,1779,0.855,46,46,46


In [11]:
# ── Query 5: High-confidence phishing by archetype (for training) ────
query_5: str = """
SELECT
    t.subject,
    t.archetype,
    t.confidence,
    t.signals,
    t.source_dataset,
    t.action_taken,
    f.feedback_label
FROM threat_log t
LEFT JOIN feedback f ON t.id = f.threat_log_id
WHERE t.verdict = 'phishing' AND t.confidence >= 0.92
ORDER BY t.confidence DESC
LIMIT 20
"""

df_q5: pd.DataFrame = pd.read_sql(query_5, engine)
print(f"Query 5: High-confidence phishing (≥0.92) — {len(df_q5)} rows")
display(df_q5)

Query 5: High-confidence phishing (≥0.92) — 20 rows


,subject,archetype,confidence,signals,source_dataset,action_taken,feedback_label
0,Avis d'imposition — Erreur détectée sur votre ...,dgfip_tax,0.990,"[""DMARC fail"", ""Suspicious URL""]",adapted_en_fr,trashed,None
1,Contrôle fiscal — Convocation DOS-0021-7659-18,dgfip_tax,0.990,"[""DMARC fail"", ""Suspicious URL""]",adapted_en_fr,trashed,false_positive
2,Avis d'imposition — Erreur détectée sur votre ...,dgfip_tax,0.990,"[""Lookalike domain"", ""PII request""]",adapted_en_fr,trashed,None
3,Mise à jour obligatoire — Espace URSSAF,urssaf_cotisation,0.990,"[""New sender"", ""Attachment suspicious""]",adapted_en_fr,trashed,None
4,Mise à jour obligatoire — Espace URSSAF,urssaf_cotisation,0.990,"[""Reply-to mismatch"", ""French impersonation""]",adapted_en_fr,trashed,None
5,Votre colis n'a pas pu être livré — DOS-8581-8...,laposte_colis,0.990,"[""URL mismatch"", ""DKIM fail""]",adapted_en_fr,trashed,None
6,Mise à jour 3D Secure obligatoire,banque_securite,0.990,"[""New sender"", ""Attachment suspicious""]",adapted_en_fr,trashed,None
7,EDF — Régularisation annuelle de votre contrat,facture_paiement,0.990,"[""DMARC fail"", ""Suspicious URL""]",adapted_en_fr,trashed,None
8,Déclaration trimestrielle — Rappel urgent,urssaf_cotisation,0.990,"[""URL mismatch"", ""DKIM fail""]",synthetic_fr,trashed,None
9,Suspension de vos droits CAF — Déclaration man...,caf_allocation,0.990,"[""DMARC fail"", ""Suspicious URL""]",synthetic_fr,trashed,None


In [12]:
# ── Query 6: Daily threat volume (time-series) ───────────────────────
query_6: str = """
SELECT
    DATE(received_at) AS day,
    COUNT(*) AS total,
    SUM(CASE WHEN verdict = 'phishing' THEN 1 ELSE 0 END) AS phishing,
    SUM(CASE WHEN verdict = 'legitimate' THEN 1 ELSE 0 END) AS legitimate,
    ROUND(AVG(confidence), 3) AS avg_confidence
FROM threat_log
WHERE received_at IS NOT NULL
GROUP BY DATE(received_at)
ORDER BY day DESC
LIMIT 30
"""

df_q6: pd.DataFrame = pd.read_sql(query_6, engine)
print(f"Query 6: Daily threat volume (last 30 days with data) — {len(df_q6)} days")
display(df_q6.head(10))

Query 6: Daily threat volume (last 30 days with data) — 30 days


,day,total,phishing,legitimate,avg_confidence
0,2026-02-28,56,47,9,0.862
1,2026-02-27,53,42,11,0.831
2,2026-02-26,53,45,8,0.867
3,2026-02-25,52,41,11,0.850
4,2026-02-24,54,43,11,0.831
5,2026-02-23,61,46,15,0.845
6,2026-02-22,59,50,9,0.853
7,2026-02-21,35,30,5,0.862
8,2026-02-20,61,51,10,0.830
9,2026-02-19,81,75,6,0.882


In [13]:
# ── Query 7: User-scoped extraction (IDOR pattern demo) ──────────────
#
# This demonstrates the security pattern: always scope queries by user_id.
# In production, user_id comes from the authenticated session — never from
# the request body.

demo_user_id: str = user_ids[0]
demo_user_name: str = users_data[0]["display_name"]

query_7: str = f"""
SELECT
    t.subject,
    t.verdict,
    t.confidence,
    t.archetype,
    t.action_taken,
    t.received_at
FROM threat_log t
WHERE t.user_id = '{demo_user_id}'
ORDER BY t.received_at DESC
LIMIT 10
"""

df_q7: pd.DataFrame = pd.read_sql(query_7, engine)
print(f"Query 7: User-scoped extraction for '{demo_user_name}' (IDOR-safe pattern)")
print(f"  User ID: {demo_user_id}")
print(f"  Results: {len(df_q7)} rows (user sees only their own data)")
display(df_q7)

Query 7: User-scoped extraction for 'Lucie Marie' (IDOR-safe pattern)
  User ID: 6211bb60-19c3-419a-9956-4c1b7eeea043
  Results: 10 rows (user sees only their own data)


,subject,verdict,confidence,archetype,action_taken,received_at
0,Remboursement fiscal en attente — Action requise,phishing,0.831,dgfip_tax,none,2026-02-28 10:23:50.428545
1,Doctolib — Rappel de rendez-vous,legitimate,0.762,health_legit,none,2026-02-28 08:12:50.428545
2,Régularisation urgente de vos cotisations — N°...,phishing,0.981,urssaf_cotisation,trashed,2026-02-28 01:15:50.428545
3,Récapitulatif mensuel — LCL,legitimate,0.657,banking_legit,none,2026-02-28 00:32:50.428545
4,Virement suspect détecté — Validation requise,phishing,0.951,banque_securite,trashed,2026-02-27 19:05:50.428545
5,Confirmation de rendez-vous — 02/03/2026,legitimate,0.563,professional_legit,none,2026-02-27 08:31:50.428545
6,"Remboursement Ameli en attente — 1 520,23 €",phishing,0.984,ameli_sante,trashed,2026-02-26 17:33:50.428545
7,Votre commande a été expédiée — DOS-1019-9676-18,legitimate,0.641,ecommerce_legit,none,2026-02-26 12:34:50.428545
8,Cotisations impayées — Mise en demeure N°-3678...,phishing,0.925,urssaf_cotisation,trashed,2026-02-26 08:30:50.428545
9,Colissimo — Nouvelle tentative de livraison,phishing,0.880,laposte_colis,trashed,2026-02-26 06:30:50.428545


## 5. Export

In [14]:
# ── Export full threat log with joins ─────────────────────────────────
export_query: str = """
SELECT
    t.id AS threat_id,
    u.display_name AS user_name,
    u.plan AS user_plan,
    t.message_id,
    t.subject,
    t.received_at,
    t.verdict,
    t.confidence,
    t.signals,
    t.archetype,
    t.source_dataset,
    t.model_version,
    t.action_taken,
    t.action_at,
    f.feedback_label,
    f.comment AS feedback_comment
FROM threat_log t
JOIN users u ON t.user_id = u.id
LEFT JOIN feedback f ON t.id = f.threat_log_id
ORDER BY t.received_at DESC
"""

df_export: pd.DataFrame = pd.read_sql(export_query, engine)

timestamp: str = datetime.now(timezone.utc).strftime("%Y%m%d")
filename: str = f"db_threat_export_{len(df_export)}_{timestamp}.csv"
output_path: Path = OUTPUT_DIR / filename

df_export.to_csv(output_path, index=False, encoding="utf-8")

size_kb: float = output_path.stat().st_size / 1024
print(f"Exported     : {output_path}")
print(f"Rows         : {len(df_export):,}")
print(f"Size         : {size_kb:.1f} KB")
print(f"Columns      : {list(df_export.columns)}")
print(f"\nVerdict distribution:")
print(df_export["verdict"].value_counts())
print(f"\nArchetype distribution (phishing):")
print(df_export[df_export["verdict"] == "phishing"]["archetype"].value_counts())
print(f"\nSource dataset:")
print(df_export["source_dataset"].value_counts())

Exported     : data/raw/db/db_threat_export_5263_20260228.csv
Rows         : 5,263
Size         : 1384.4 KB
Columns      : ['threat_id', 'user_name', 'user_plan', 'message_id', 'subject', 'received_at', 'verdict', 'confidence', 'signals', 'archetype', 'source_dataset', 'model_version', 'action_taken', 'action_at', 'feedback_label', 'feedback_comment']

Verdict distribution:
verdict
phishing      4400
legitimate     863
Name: count, dtype: int64

Archetype distribution (phishing):
archetype
dgfip_tax               573
urssaf_cotisation       568
banque_securite         568
laposte_colis           564
ameli_sante             550
franceconnect_id        485
caf_allocation          469
facture_paiement        462
bec_autoentrepreneur    161
Name: count, dtype: int64

Source dataset:
source_dataset
synthetic_fr     2863
adapted_en_fr    2400
Name: count, dtype: int64


## 6. Summary

### What this notebook demonstrates (C1)

| Criterion | Evidence |
|-----------|----------|
| **Source type** | Base de données (SQLite / SQL) |
| **Schema** | Canonical Sicurre schema from `data-design.md` |
| **ORM** | SQLAlchemy with dialect abstraction (SQLite dev ↔ PostgreSQL prod) |
| **Real data** | Seeded from culturally-adapted (NB10) + synthetic (NB11) French emails |
| **Users** | 10 Faker(fr_FR) auto-entrepreneur profiles |
| **Queries** | 7 analytical queries: aggregation, JOIN, filtering, GROUP BY, time-series, user-scoped |
| **IDOR demo** | Query 7 shows user-scoped extraction (security pattern) |
| **Output** | CSV export of joined threat data |

### Data lineage

| Source | Rows | Description |
|--------|------|-------------|
| `adapted_en_fr` | ~2 400 | EN→FR cultural adaptation (notebook 10) |
| `synthetic_fr` | ~3 000 | Synthetic French generation (notebook 11) |
| **Total** | **~5 400** | All French, all labeled, all traceable |

### Dialect abstraction (ADR-0004)

The same ORM models used here work against Neon PostgreSQL in production.
The only change is the connection string:

```python
# Dev (this notebook)
DB_URL = "sqlite:///data/raw/db/sicurre_dev.db"

# Prod (Cloud Run)
DB_URL = os.environ["DATABASE_URL"]  # postgresql+asyncpg://...
```

### Defence talking point

> *"The database extraction demonstrates the full SQL pipeline: schema creation,*
> *data insertion from upstream sources (culturally-adapted + synthetic French*
> *phishing emails), analytical queries with JOINs across 4 tables, and CSV export.*
> *The data is not random — it comes from our EN→FR adaptation pipeline and*
> *synthetic generation, seeded with 10 realistic Faker(fr_FR) auto-entrepreneur*
> *profiles. The same SQLAlchemy ORM works on SQLite (dev) and Neon PostgreSQL*
> *(prod) with zero code changes."*